In [1]:
import pandas as pd
import numpy as np
from sklearn.ensemble import HistGradientBoostingRegressor

# Carregar os dados

df = pd.read_csv("data_core_daily_v1.csv")
df["data"] = pd.to_datetime(df["data"])
df = df.sort_values(["CFGCENTROID","data"])


In [2]:
# Criar features mínimas necessárias
df["dow"] = df["data"].dt.dayofweek
df["week"] = df["data"].dt.isocalendar().week.astype(int)
df["month"] = df["data"].dt.month
df["is_saturday"] = (df["dow"] == 5).astype(int)

if "is_holiday" not in df.columns:
    df["is_holiday"] = 0

# Lags
for lag in [1,7,14,28]:
    df[f"lag_{lag}"] = (
        df.groupby("CFGCENTROID")["y_inspecoes"]
        .shift(lag)
    )

# Rolling
df["roll7_mean"] = (
    df.groupby("CFGCENTROID")["y_inspecoes"]
    .shift(1)
    .rolling(7)
    .mean()
    .reset_index(level=0, drop=True)
)

df["roll28_mean"] = (
    df.groupby("CFGCENTROID")["y_inspecoes"]
    .shift(1)
    .rolling(28)
    .mean()
    .reset_index(level=0, drop=True)
)

df_model = df.dropna().copy()

In [3]:
cutoff = df_model["data"].max() - pd.DateOffset(months=3)

train = df_model[df_model["data"] <= cutoff]
test  = df_model[df_model["data"] > cutoff]

features_daily = [
    "CFGCENTROID",
    "dow",
    "week",
    "month",
    "is_saturday",
    "is_holiday",
    "lag_1",
    "lag_7",
    "lag_14",
    "lag_28",
    "roll7_mean",
    "roll28_mean"
]

X_train = train[features_daily]
y_train = train["y_inspecoes"]

X_test = test[features_daily]
y_test = test["y_inspecoes"]

In [4]:
#Treinar modelo diário

model_daily = HistGradientBoostingRegressor(
    max_depth=6,
    learning_rate=0.05,
    max_iter=400,
    random_state=42
)

model_daily.fit(X_train, y_train)

pred_daily = model_daily.predict(X_test)
pred_daily = np.clip(pred_daily, 0, None)

In [5]:
#WAPE diário
wape_daily = np.sum(np.abs(y_test - pred_daily)) / np.sum(y_test)
print("WAPE diário:", round(wape_daily,4))

WAPE diário: 0.1114


In [6]:
# Agregar para semanal:
daily_test = test.copy()
daily_test["pred"] = pred_daily

daily_test["year"] = daily_test["data"].dt.isocalendar().year
daily_test["week"] = daily_test["data"].dt.isocalendar().week

weekly_from_daily = (
    daily_test
    .groupby(["CFGCENTROID","year","week"])
    .agg(
        real=("y_inspecoes","sum"),
        pred=("pred","sum")
    )
    .reset_index()
)

weekly_from_daily["abs_err"] = abs(weekly_from_daily["real"] - weekly_from_daily["pred"])

wape_weekly_from_daily = (
    weekly_from_daily["abs_err"].sum() /
    weekly_from_daily["real"].sum()
)

print("WAPE semanal (agregado do modelo diário):", round(wape_weekly_from_daily,4))

WAPE semanal (agregado do modelo diário): 0.054


In [7]:
from sklearn.linear_model import Ridge
from sklearn.ensemble import RandomForestRegressor, HistGradientBoostingRegressor
import numpy as np
import pandas as pd

models = {
    "Ridge": Ridge(alpha=1.0),
    "RandomForest": RandomForestRegressor(
        n_estimators=200,
        max_depth=10,
        random_state=42,
        n_jobs=-1
    ),
    "HGB": HistGradientBoostingRegressor(
        max_depth=6,
        learning_rate=0.05,
        max_iter=400,
        random_state=42
    )
}

results = []

for name, model in models.items():
    
    model.fit(X_train, y_train)
    pred = model.predict(X_test)
    pred = np.clip(pred, 0, None)
    
    wape_daily = np.sum(np.abs(y_test - pred)) / np.sum(y_test)
    
    # Agregação semanal
    tmp = test.copy()
    tmp["pred"] = pred
    tmp["year"] = tmp["data"].dt.isocalendar().year
    tmp["week"] = tmp["data"].dt.isocalendar().week
    
    weekly_tmp = (
        tmp.groupby(["CFGCENTROID","year","week"])
        .agg(real=("y_inspecoes","sum"),
             pred=("pred","sum"))
        .reset_index()
    )
    
    weekly_tmp["abs_err"] = abs(weekly_tmp["real"] - weekly_tmp["pred"])
    
    wape_weekly = (
        weekly_tmp["abs_err"].sum() /
        weekly_tmp["real"].sum()
    )
    
    results.append({
        "Model": name,
        "WAPE Diário": wape_daily,
        "WAPE Semanal": wape_weekly
    })

pd.DataFrame(results).sort_values("WAPE Diário")

,Model,WAPE Diário,WAPE Semanal
2,HGB,0.111361,0.054002
1,RandomForest,0.113554,0.047493
0,Ridge,0.130073,0.053902


In [8]:
from sklearn.ensemble import RandomForestRegressor

# Treinar RF
rf = RandomForestRegressor(
    n_estimators=300,
    max_depth=12,
    random_state=42,
    n_jobs=-1
)

rf.fit(X_train, y_train)
pred_rf = np.clip(rf.predict(X_test), 0, None)

# Treinar HGB
hgb = HistGradientBoostingRegressor(
    max_depth=6,
    learning_rate=0.05,
    max_iter=400,
    random_state=42
)

hgb.fit(X_train, y_train)
pred_hgb = np.clip(hgb.predict(X_test), 0, None)

# Ensemble (média simples)
pred_ensemble = (pred_hgb + pred_rf) / 2

# WAPE diário
wape_daily_ens = np.sum(np.abs(y_test - pred_ensemble)) / np.sum(y_test)
print("WAPE Diário Ensemble:", round(wape_daily_ens,4))

# Agregar semanal
tmp = test.copy()
tmp["pred"] = pred_ensemble
tmp["year"] = tmp["data"].dt.isocalendar().year
tmp["week"] = tmp["data"].dt.isocalendar().week

weekly_tmp = (
    tmp.groupby(["CFGCENTROID","year","week"])
    .agg(real=("y_inspecoes","sum"),
         pred=("pred","sum"))
    .reset_index()
)

weekly_tmp["abs_err"] = abs(weekly_tmp["real"] - weekly_tmp["pred"])
wape_weekly_ens = weekly_tmp["abs_err"].sum() / weekly_tmp["real"].sum()

print("WAPE Semanal Ensemble:", round(wape_weekly_ens,4))

WAPE Diário Ensemble: 0.11
WAPE Semanal Ensemble: 0.0488


In [9]:
center_volume = (
    df_model.groupby("CFGCENTROID")["y_inspecoes"]
    .mean()
    .reset_index()
)

median_vol = center_volume["y_inspecoes"].median()

large_centers = center_volume[
    center_volume["y_inspecoes"] > median_vol
]["CFGCENTROID"].valuestrain_large = train[train["CFGCENTROID"].isin(large_centers)]
test_large  = test[test["CFGCENTROID"].isin(large_centers)]

X_train_l = train_large[features_daily]
y_train_l = train_large["y_inspecoes"]

X_test_l = test_large[features_daily]
y_test_l = test_large["y_inspecoes"]

model_large = HistGradientBoostingRegressor(
    max_depth=6,
    learning_rate=0.05,
    max_iter=400,
    random_state=42
)

model_large.fit(X_train_l, y_train_l)
pred_large = np.clip(model_large.predict(X_test_l), 0, None)

wape_large = np.sum(np.abs(y_test_l - pred_large)) / np.sum(y_test_l)
print("WAPE centros grandes:", round(wape_large,4))

NameError: name 'large_centers' is not defined